# HACCP 도마-식재료 교차오염 판정 (YOLO11)

**4차(마지막) 데이터 보강 — 도마_초록 사진 4장을 추가해 342장으로 재학습합니다.**

## 클래스 구성 (8개, `classes.txt` 순서 그대로 class id 0~7)
| id | 클래스 | 1차(111) | 2차(181) | 3차(338) | 4차(342) |
|---|---|---|---|---|---|
| 0 | `도마_빨강` | 55 | 55 | 100 | 100 |
| 1 | `도마_파랑` | 32 | 41 | 87 | 87 |
| 2 | `도마_초록` | 23 | 30 | 78 | 82 |
| 3 | `재료_육류` | 54 | 107 | 169 | 169 |
| 4 | `재료_생선` | 48 | 79 | 127 | 127 |
| 5 | `재료_채소` | 91 | 96 | 182 | 182 |
| 6 | `손` | 49 | 63 | 76 | 76 |
| 7 | `칼` | 62 | 72 | 76 | 76 |

3차 학습에서 유일하게 남았던 약점(도마_초록 탐지 실패 2건, `green_board_0001`/`green_board_only_0006`)을 보강하기 위해
도마_초록 사진 4장만 추가했습니다. 이번이 마지막 데이터 보강 라운드입니다.

## 정상/위반 판정 규칙 (모델이 아니라 코드가 계산)
| 도마 색 | 정상으로 인정되는 재료 | 그 외 재료가 감지되면 |
|---|---|---|
| 빨강 | 육류 | 위반_교차오염 |
| 파랑 | 생선 | 위반_교차오염 |
| 초록 | 채소 | 위반_교차오염 |

YOLO는 "도마 색"과 "재료 종류"만 탐지하고, 정상/위반은 노트북 뒤쪽의 규칙 함수가 계산합니다.

## 데이터 분할
- train 290장 → **750장** (도마 라벨 있는 이미지에 조명 변형본 2장씩 추가) / val 51장
- 추가된 `*_litaug1/2` 이미지는 밝기 감소+푸르스름한 색조+물기 반사 하이라이트를 합성한 것으로, 실제 도마 위치/모양은 원본과 동일합니다 (라벨 좌표 재사용)
- val 51장은 3차와 동일한 랜덤 시드로 뽑혀서, 3차 결과와 비교적 공정하게 비교할 수 있습니다 (새로 추가된 4장 중 val로 뽑힌 것이 있으면 인스턴스 수가 약간 다를 수 있음)

Google Colab에서 런타임을 **GPU(T4)**로 설정한 뒤 위에서부터 순서대로 실행하세요.

## 0) GPU 확인 (T4가 잡혔는지 확인)

In [ ]:
!nvidia-smi

## 1) ultralytics 설치

In [ ]:
!pip install -U ultralytics

## 2) 데이터셋 업로드

로컬에서 만든 `haccp_dataset.zip`(약 67MB, `images/train`, `images/val`, `labels/train`, `labels/val`, `data.yaml` 포함)을 업로드합니다.

**옵션 A — 직접 업로드 (권장, 이 정도 용량이면 충분히 빠름)**
아래 셀을 실행하면 파일 선택창이 뜹니다. `haccp_dataset.zip`을 선택하세요.

**옵션 B — Google Drive 경유**
Drive에 미리 올려뒀다면 옵션 A 대신 옵션 B 셀의 주석을 풀어서 쓰세요.

In [ ]:
# ── 옵션 A: 직접 업로드 ──
from google.colab import files
uploaded = files.upload()  # haccp_dataset.zip 선택
!unzip -o haccp_dataset.zip -d /content/

# ── 옵션 B: Google Drive 경유 (필요시 주석 해제 후 옵션 A 대신 사용) ──
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -o "/content/drive/MyDrive/본인_경로/haccp_dataset.zip" -d /content/

## 3) data.yaml 확인

In [ ]:
%cat data.yaml

## 4) 학습 (Train)

- `model=yolo11s.pt` : 처음엔 s(=small)로 시작. 과적합 조짐(train loss는 계속 내려가는데 val loss가 오히려 오름)이 보이면 `yolo11n.pt`(nano)로 낮춰서 다시 시도하세요.
- COCO 사전학습 가중치에서 시작 — 데이터가 적을 때 훨씬 잘 수렴합니다.
- `patience=50` : 50 에폭 동안 val 성능 개선이 없으면 자동으로 조기 종료합니다 (작은 데이터셋에서 불필요하게 300 에폭 다 채우며 과적합되는 것 방지).
- `imgsz=640` (416에서 상향): 칼/대파처럼 얇고 작은 물체가 많아서, 해상도를 높여야 디테일이 덜 뭉개집니다. T4 메모리로 충분히 감당됩니다.
- `batch=-1`: GPU 메모리의 약 60%를 기준으로 배치 크기를 자동으로 잡아줍니다. imgsz를 올린 상태에서 직접 batch를 가늠하는 것보다 안전합니다.
- `cache=True`: 데이터셋이 크지 않아 이미지를 캐싱해두면 매 에폭 디스크 재읽기 시간을 아낄 수 있습니다.
- `seed=42`: 결과 재현/비교를 위해 시드를 고정합니다.

In [ ]:
%%time
!yolo detect train     data=data.yaml     model=yolo11s.pt     imgsz=640     epochs=300     patience=50     batch=-1     cache=True     seed=42     name=haccp_yolo11s

## 5) TensorBoard로 학습 곡선 보기

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/detect

## 6) val 이미지에 대해 탐지만 먼저 확인 (도마/재료/손/칼 박스가 잘 잡히는지)

In [ ]:
!yolo detect predict     model=runs/detect/haccp_yolo11s/weights/best.pt     imgsz=640     conf=0.4     source=images/val     save=True

## 7) 학습 결과 그래프 (results.png)

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename='runs/detect/haccp_yolo11s/results.png', width=1000)

## 8) 검증 배치 예측 결과 (Ground Truth vs Prediction)

In [ ]:
print("VALIDATION BATCH - PREDICTION 예시:")
IPyImage(filename='runs/detect/haccp_yolo11s/val_batch0_pred.jpg', width=900)

## 9) 증강된(Augmented) 학습 배치 예시

In [ ]:
print("AUGMENTED TRAINING DATA 예시:")
IPyImage(filename='runs/detect/haccp_yolo11s/train_batch0.jpg', width=900)

## 10) 정상/위반 최종 판정 (탐지 결과 + 규칙 조합)

- YOLO는 이미지에서 `도마_*`, `재료_*` 박스를 각각 탐지합니다.
- 코드가 "가장 확신도(confidence) 높은 도마 색"과 "가장 확신도 높은 재료 종류"를 뽑아서,
  위쪽 표의 규칙과 비교해 최종적으로 `정상_OOO` 또는 `위반_교차오염`을 출력합니다.
- 도마를 못 찾으면 `판정불가`, 도마는 찾았는데 재료가 없으면 `재료없음`(빈 도마 상태)으로 따로 구분합니다.
- 억지로 추측해서 정상/위반을 단정하지 않습니다.

In [ ]:
from ultralytics import YOLO
import glob, os

RUN_NAME = "haccp_yolo11s"
MODEL_PATH = f"runs/detect/{RUN_NAME}/weights/best.pt"

# 도마 색상 -> 정상으로 인정되는 재료. 이 외의 재료가 감지되면 위반.
NORMAL_MATCH = {
    "빨강": "육류",
    "파랑": "생선",
    "초록": "채소",
}

model = YOLO(MODEL_PATH)

def judge_image(result):
    """YOLO 탐지 결과(이미지 1장)를 받아 최종 정상/위반 판정 문자열을 반환"""
    names = result.names
    best_board, best_board_conf = None, -1.0
    best_ing, best_ing_conf = None, -1.0

    for box in result.boxes:
        cls_name = names[int(box.cls)]
        conf = float(box.conf)
        if cls_name.startswith("도마_") and conf > best_board_conf:
            best_board, best_board_conf = cls_name.split("_", 1)[1], conf
        elif cls_name.startswith("재료_") and conf > best_ing_conf:
            best_ing, best_ing_conf = cls_name.split("_", 1)[1], conf

    if best_board is None:
        return "판정불가 (도마를 못 찾음)"

    if best_ing is None:
        return f"재료없음  (도마:{best_board} conf={best_board_conf:.2f})"

    if NORMAL_MATCH.get(best_board) == best_ing:
        return f"정상_{best_ing}  (도마:{best_board} conf={best_board_conf:.2f}, 재료:{best_ing} conf={best_ing_conf:.2f})"
    return f"위반_교차오염  (도마:{best_board} conf={best_board_conf:.2f}, 재료:{best_ing} conf={best_ing_conf:.2f})"

CONF_THRESHOLD = 0.4  # 여기 값만 바꾸면 6/10/12번 셀 전부에 일괄 반영됩니다

val_images = sorted(glob.glob("images/val/*.*"))
results = model.predict(source=val_images, imgsz=640, conf=CONF_THRESHOLD, save=True)

for path, result in zip(val_images, results):
    print(f"{os.path.basename(path):40s} -> {judge_image(result)}")

## 11) 판정 결과 집계 (정상/위반/판정불가 개수)

In [ ]:
from collections import Counter

verdict_counts = Counter()
for result in results:
    verdict = judge_image(result)
    key = verdict.split("  ")[0].split(" (")[0]
    verdict_counts[key] += 1

for k, v in verdict_counts.items():
    print(f"{k}: {v}장")

## 12) 발표용 예측 결과 이미지 만들기 (맞은 예시 / 틀린 예시)

7번 슬라이드("딥러닝 예측 결과")에 쓸 이미지를 자동으로 만듭니다.
- 각 val 이미지의 **정답**(라벨 txt에서 직접 읽음)과 **모델 예측**(10번 셀 결과)을 비교합니다.
- 이미지 아래에 "예측: OOO" / "정답: OOO" 텍스트를 같이 그려서, 맞은 것과 틀린 것을 각각 `presentation_examples/correct`, `presentation_examples/wrong` 폴더에 저장합니다.
- 발표 자료엔 **틀린 예시가 특히 중요**합니다 — 왜 틀렸는지 분석까지 같이 준비하세요 (예: 조명, 가려짐, 색상 유사 등).

In [ ]:
# 한글이 깨지지 않도록 폰트 설치 (최초 1회)
!apt-get -qq install -y fonts-nanum > /dev/null


In [ ]:
import os
from PIL import Image, ImageDraw, ImageFont

LABELS_VAL_DIR = "labels/val"

try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 22)
except Exception:
    font = ImageFont.load_default()


def get_ground_truth_category(image_path, class_names):
    """val 라벨(txt)에서 실제 도마색/재료를 읽어, judge_image와 같은 규칙으로 '정답 카테고리'를 계산"""
    name = os.path.splitext(os.path.basename(image_path))[0]
    label_path = os.path.join(LABELS_VAL_DIR, name + ".txt")
    if not os.path.exists(label_path):
        return "라벨없음"

    board, ing = None, None
    with open(label_path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_name = class_names[int(parts[0])]
            if cls_name.startswith("도마_") and board is None:
                board = cls_name.split("_", 1)[1]
            elif cls_name.startswith("재료_") and ing is None:
                ing = cls_name.split("_", 1)[1]

    if board is None:
        return "판정불가"
    if ing is None:
        return "재료없음"
    if NORMAL_MATCH.get(board) == ing:
        return f"정상_{ing}"
    return "위반_교차오염"


def category_of(verdict_str):
    """judge_image()가 반환한 '정상_생선  (도마:... conf=...)' 같은 문자열에서 카테고리만 추출"""
    return verdict_str.split("  ")[0].split(" (")[0]


os.makedirs("presentation_examples/correct", exist_ok=True)
os.makedirs("presentation_examples/wrong", exist_ok=True)

correct_files, wrong_files = [], []
for path, result in zip(val_images, results):
    pred_verdict = judge_image(result)
    pred_cat = category_of(pred_verdict)
    true_cat = get_ground_truth_category(path, model.names)
    is_correct = (pred_cat == true_cat)

    im = Image.open(path).convert("RGB")
    im.thumbnail((500, 500))
    canvas = Image.new("RGB", (im.width, im.height + 55), "white")
    canvas.paste(im, (0, 0))
    draw = ImageDraw.Draw(canvas)
    draw.text((5, im.height + 3), f"예측: {pred_cat}", fill=(0, 0, 200), font=font)
    draw.text((5, im.height + 28), f"정답: {true_cat}", fill=(0, 0, 0), font=font)

    out_dir = "presentation_examples/correct" if is_correct else "presentation_examples/wrong"
    canvas.save(os.path.join(out_dir, os.path.basename(path)))
    (correct_files if is_correct else wrong_files).append(os.path.basename(path))

print(f"맞은 예시: {len(correct_files)}장 -> presentation_examples/correct/")
print(f"틀린 예시: {len(wrong_files)}장 -> presentation_examples/wrong/")
if wrong_files:
    print("\n틀린 예시 목록 (원인 분석해서 슬라이드에 같이 적으세요):")
    for f in wrong_files:
        print(" ", f)


### 틀린 예시 한눈에 보기 (발표 슬라이드용 그리드 이미지)

In [ ]:
import glob
from PIL import Image as PILImage

wrong_paths = sorted(glob.glob("presentation_examples/wrong/*.*"))
COLS = 4
THUMB = 260

if wrong_paths:
    rows = (len(wrong_paths) + COLS - 1) // COLS
    sheet = PILImage.new("RGB", (COLS * THUMB, rows * THUMB), "white")
    for i, p in enumerate(wrong_paths):
        im = PILImage.open(p)
        im.thumbnail((THUMB, THUMB))
        r, c = divmod(i, COLS)
        sheet.paste(im, (c * THUMB, r * THUMB))
    sheet.save("presentation_examples/wrong_grid.jpg")
    print(f"{len(wrong_paths)}장 -> presentation_examples/wrong_grid.jpg (이 파일을 다운로드해서 발표자료에 쓰세요)")
    display(sheet)
else:
    print("틀린 예시가 없습니다 — val셋 기준으로는 전부 맞았다는 뜻입니다.")


## 13) (선택) 배포용 포맷으로 내보내기 (Export)

In [ ]:
!yolo export model=runs/detect/haccp_yolo11s/weights/best.pt format=onnx

## 13-1) 안드로이드 실시간 데모용 TFLite 내보내기

**재학습 필요 없습니다.** 이미 로컬에 있는 `best.pt`를 그대로 업로드해서 변환만 하면 됩니다 (이 섹션은 새 Colab 세션에서 위 학습 셀들을 실행하지 않고 여기부터 바로 실행 가능).

- Windows(로컬 PC)에서는 TFLite 변환 자체가 안 됩니다 (Ultralytics의 TFLite export가 Linux x86/macOS에서만 지원됨). Colab은 Linux라 여기서 변환합니다.
- 변환 후 `best_saved_model/best_float32.tflite` 파일을 다운로드해서, 폰에 **Ultralytics 공식 안드로이드 앱**(Play Store)을 설치하고 "커스텀 모델 불러오기"로 이 파일을 얹으면 됩니다.
- 주의: 이 앱은 도마/재료/손/칼 박스 탐지까지만 화면에 보여주고, `judge_image()`의 정상/위반 최종 판정 로직은 적용되지 않습니다 — 발표 때는 "도마_빨강 + 재료_채소가 동시에 잡히면 위반"이라고 직접 설명을 곁들이면 됩니다.

In [ ]:
!pip install -U ultralytics

from google.colab import files
uploaded_weights = files.upload()  # best.pt 선택 (로컬 학습결과사진/마지막학습/.../weights/best.pt)
WEIGHTS_PATH = list(uploaded_weights.keys())[0]
print("업로드된 가중치:", WEIGHTS_PATH)

In [ ]:
from ultralytics import YOLO
import glob

model_export = YOLO(WEIGHTS_PATH)
model_export.export(format="tflite", imgsz=640)

tflite_path = glob.glob("*_saved_model/*float32.tflite")
if tflite_path:
    print("변환 완료:", tflite_path[0])
    files.download(tflite_path[0])
else:
    print("tflite 파일을 못 찾았습니다. 위 export 로그에서 실제 저장 경로를 확인하세요.")

## 14) 영상(비디오)으로 테스트 — 실시간 교차오염 감시 시뮬레이션

이 프로젝트의 목표는 "사진 한 장 분류"가 아니라 **주방 카메라 영상을 실시간으로 보면서 교차오염을 감시하는 것**이라, 정지 이미지 val셋 결과만으로는 부족합니다.
직접 촬영한(또는 구한) 짧은 영상(10~30초 정도 권장)으로 실제 동작을 확인하세요.

- 프레임마다 도마/재료를 탐지하고, 그 프레임의 정상/위반 판정을 화면 상단에 실시간으로 표시합니다.
- **판정 안정화(다수결 스무딩)**: 프레임 하나하나는 독립 판정이라 순간적인 오탐으로 판정이 깜빡일 수 있습니다. 최근 `SMOOTH_WINDOW`(기본 15프레임 ≈ 0.5~1초) 판정 중 다수결로 화면에 표시할 최종 판정을 정합니다.
- **위반 기록(감사 로그)**: HACCP 규정은 기록 보관이 핵심입니다. 다수결 판정이 "위반_교차오염"으로 새로 진입한 시점부터, 벗어나는 시점까지를 하나의 이벤트로 보고 `violation_log.csv`에 자동 기록합니다.
  - 시각은 `datetime.now()` 기준 **실제 시:분:초**로 기록됩니다. 지금(업로드된 영상 파일을 배치로 처리)은 이 시각이 "처리한 시점"일 뿐이지만, 나중에 이 코드가 웹캠 실시간 입력으로 바뀌면 그대로 "실제 위반 발생 시각"이 되도록 미리 이렇게 설계했습니다.
- 결과 영상은 `runs/detect/video_result.mp4`, 위반 기록은 `violation_log.csv`에 저장됩니다.
- 영상이 길면 처리 시간이 늘어나니, 처음엔 짧은 영상으로 먼저 테스트해보세요.

### 14-0) (이 세션에서 학습을 안 했다면) `best.pt`만 올려서 독립 실행

위 학습/데이터셋 셀들을 건너뛰고 **`best.pt`만 업로드해서 바로 영상 테스트**를 하려면 이 셀부터 실행하세요.
(이미 같은 세션에서 학습을 마쳤다면 `model`, `judge_image`, `CONF_THRESHOLD`가 이미 정의되어 있으니 이 셀은 건너뛰어도 됩니다.)

In [ ]:
!pip install -U ultralytics

from google.colab import files
from ultralytics import YOLO
import torch

uploaded_weights = files.upload()  # best.pt 선택 (로컬 학습결과사진/마지막학습/.../weights/best.pt)
MODEL_PATH = list(uploaded_weights.keys())[0]
print("업로드된 가중치:", MODEL_PATH)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("경고: GPU를 못 찾았습니다 (CPU로 실행됨, 영상 처리가 훨씬 느립니다).")
    print("   -> 상단 메뉴 [런타임 > 런타임 유형 변경 > T4 GPU]로 바꾸고 다시 연결한 뒤 이 셀부터 재실행하세요.")
else:
    print(f"GPU 사용: {torch.cuda.get_device_name(0)}")

model = YOLO(MODEL_PATH)

# 도마 색상 -> 정상으로 인정되는 재료. 이 외의 재료가 감지되면 위반.
NORMAL_MATCH = {
    "빨강": "육류",
    "파랑": "생선",
    "초록": "채소",
}
CONF_THRESHOLD = 0.4


def judge_image(result):
    """YOLO 탐지 결과(이미지 1장)를 받아 최종 정상/위반 판정 문자열을 반환"""
    names = result.names
    best_board, best_board_conf = None, -1.0
    best_ing, best_ing_conf = None, -1.0

    for box in result.boxes:
        cls_name = names[int(box.cls)]
        conf = float(box.conf)
        if cls_name.startswith("도마_") and conf > best_board_conf:
            best_board, best_board_conf = cls_name.split("_", 1)[1], conf
        elif cls_name.startswith("재료_") and conf > best_ing_conf:
            best_ing, best_ing_conf = cls_name.split("_", 1)[1], conf

    if best_board is None:
        return "판정불가 (도마를 못 찾음)"
    if best_ing is None:
        return f"재료없음  (도마:{best_board} conf={best_board_conf:.2f})"
    if NORMAL_MATCH.get(best_board) == best_ing:
        return f"정상_{best_ing}  (도마:{best_board} conf={best_board_conf:.2f}, 재료:{best_ing} conf={best_ing_conf:.2f})"
    return f"위반_교차오염  (도마:{best_board} conf={best_board_conf:.2f}, 재료:{best_ing} conf={best_ing_conf:.2f})"


def category_of(verdict_str):
    """judge_image()가 반환한 '정상_생선  (도마:... conf=...)' 같은 문자열에서 카테고리만 추출"""
    return verdict_str.split("  ")[0].split(" (")[0]

In [ ]:
from google.colab import files
uploaded_video = files.upload()  # 테스트할 mp4 영상 선택
VIDEO_PATH = list(uploaded_video.keys())[0]
print("업로드된 영상:", VIDEO_PATH)

In [ ]:
import cv2
import csv
import os
import subprocess
import torch
import numpy as np
from collections import deque, Counter
from datetime import datetime, timezone, timedelta
from PIL import Image, ImageDraw, ImageFont

KST = timezone(timedelta(hours=9))  # Colab 서버는 UTC라서 한국 시간(KST)으로 명시 변환

OUT_VIDEO_PATH = "runs/detect/video_result.mp4"
LOG_CSV_PATH = "violation_log.csv"

os.makedirs(os.path.dirname(OUT_VIDEO_PATH), exist_ok=True)  # best.pt만 올려 학습 없이 실행할 땐 이 폴더가 없어서 미리 만들어줘야 함

SMOOTH_WINDOW = 15  # 최근 몇 프레임으로 다수결을 낼지 (약 0.5~1초 분량)

NANUM_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(NANUM_PATH):
    # 12번 섹션을 안 거치고 14번부터 바로 실행한 경우, 한글 폰트가 아직 없어서 여기서 설치
    # (설치 안 되어 있으면 PIL이 기본 비트맵 폰트로 폴백해서 판정 텍스트가 작게 나옴)
    print("나눔고딕 폰트 설치 중...")
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], stdout=subprocess.DEVNULL)

try:
    font_video = ImageFont.truetype(NANUM_PATH, 32)
except Exception as e:
    print(f"폰트 로드 실패, 기본 폰트로 대체합니다: {e}")
    font_video = ImageFont.load_default()

# 판정 카테고리별 표시 색상 (PIL 기준 RGB)
COLOR_NORMAL = (0, 170, 0)
COLOR_VIOLATION = (220, 0, 0)
COLOR_NEUTRAL = (120, 120, 120)


def verdict_color(category):
    if category.startswith("정상"):
        return COLOR_NORMAL
    if category == "위반_교차오염":
        return COLOR_VIOLATION
    return COLOR_NEUTRAL


def extract_board_ing(result):
    """judge_image()와 동일한 규칙으로, 로그 기록용 도마색/재료명만 뽑아낸다."""
    names = result.names
    best_board, best_board_conf = None, -1.0
    best_ing, best_ing_conf = None, -1.0
    for box in result.boxes:
        cls_name = names[int(box.cls)]
        conf = float(box.conf)
        if cls_name.startswith("도마_") and conf > best_board_conf:
            best_board, best_board_conf = cls_name.split("_", 1)[1], conf
        elif cls_name.startswith("재료_") and conf > best_ing_conf:
            best_ing, best_ing_conf = cls_name.split("_", 1)[1], conf
    return best_board, best_ing


if "DEVICE" not in dir():
    DEVICE = 0 if torch.cuda.is_available() else "cpu"  # 14-0을 안 거쳤다면 여기서 한 번 더 확인

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 24
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"영상 정보: {frame_w}x{frame_h}, {fps:.1f}fps, 총 {total_frames}프레임")
print(f"추론 장치: {'GPU (' + torch.cuda.get_device_name(0) + ')' if DEVICE != 'cpu' else 'CPU (느릴 수 있습니다 - 런타임을 GPU로 바꾸는 걸 권장)'}")

writer = cv2.VideoWriter(OUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))

history = deque(maxlen=SMOOTH_WINDOW)
violation_log = []  # 완료된 위반 이벤트들
violation_active = False
v_start_dt = v_start_board = v_start_ing = None

frame_idx = 0
smoothed_history = []  # 스무딩된(=화면에 실제로 표시된) 판정 기록 -> 집계용
while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.predict(source=frame, imgsz=640, conf=CONF_THRESHOLD, device=DEVICE, verbose=False)[0]
    plotted = result.plot()  # 박스가 그려진 프레임 (BGR)

    raw_category = category_of(judge_image(result))
    history.append(raw_category)
    smoothed = Counter(history).most_common(1)[0][0]  # 최근 SMOOTH_WINDOW프레임 다수결
    smoothed_history.append(smoothed)
    color = verdict_color(smoothed)

    # --- 위반 이벤트 시작/종료 감지 (스무딩된 판정 기준) ---
    # 실제 시각(KST 기준 datetime.now())으로 기록 -> 지금은 "영상 파일을 처리한 시각"일 뿐이지만,
    # 나중에 이 코드가 웹캠 실시간 입력으로 바뀌면 이 시각이 곧 "실제 위반 발생 시각"이 됩니다.
    if smoothed == "위반_교차오염" and not violation_active:
        violation_active = True
        v_start_dt = datetime.now(KST)
        v_start_board, v_start_ing = extract_board_ing(result)
    elif smoothed != "위반_교차오염" and violation_active:
        violation_active = False
        v_end_dt = datetime.now(KST)
        violation_log.append({
            "start_time": v_start_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "end_time": v_end_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "duration_sec": round((v_end_dt - v_start_dt).total_seconds(), 2),
            "board_color": v_start_board,
            "ingredient": v_start_ing,
        })

    pil_img = Image.fromarray(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_img)
    draw.rectangle([(0, 0), (frame_w, 55)], fill=(255, 255, 255))
    draw.text((10, 8), f"판정: {smoothed}", fill=color, font=font_video)

    out_frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    writer.write(out_frame)

    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f"  처리 중... {frame_idx}/{total_frames}")

# 영상이 위반 상태로 끝난 경우, 마지막 프레임에서 이벤트를 마감 처리
if violation_active:
    v_end_dt = datetime.now(KST)
    violation_log.append({
        "start_time": v_start_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "end_time": v_end_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "duration_sec": round((v_end_dt - v_start_dt).total_seconds(), 2),
        "board_color": v_start_board,
        "ingredient": v_start_ing,
    })

cap.release()
writer.release()

with open(LOG_CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
    fieldnames = ["start_time", "end_time", "duration_sec", "board_color", "ingredient"]
    csv_writer = csv.DictWriter(f, fieldnames=fieldnames)
    csv_writer.writeheader()
    csv_writer.writerows(violation_log)

print(f"\n완료: {frame_idx}프레임 처리 -> {OUT_VIDEO_PATH}")
print(f"위반 이벤트 {len(violation_log)}건 기록 -> {LOG_CSV_PATH}")
for row in violation_log:
    print(f"  [{row['start_time']} ~ {row['end_time']}, {row['duration_sec']}s] "
          f"도마:{row['board_color']} 재료:{row['ingredient']}")

print("\n프레임별 판정 분포 (스무딩 후):")
for cat, cnt in Counter(smoothed_history).most_common():
    print(f"  {cat}: {cnt}프레임 ({cnt/max(1,frame_idx)*100:.1f}%)")

### 결과 영상 + 위반 로그 다운로드

아래 셀을 실행하면 `video_result.mp4`와 `violation_log.csv`가 같이 다운로드됩니다.
- 영상은 발표 때 재생해서 "정지 이미지 분류"가 아니라 "실시간 감시"라는 프로젝트 목표를 보여주는 용도
- CSV는 "HACCP 규정이 요구하는 기록 보관"까지 시스템이 자동으로 처리한다는 걸 보여주는 용도 (엑셀로 바로 열림, utf-8-sig 인코딩)

In [ ]:
from google.colab import files
files.download("runs/detect/video_result.mp4")
files.download("violation_log.csv")